In [8]:
import json
import pandas as pd
import os

# Step 1: Get all JSON files
json_files = [f for f in os.listdir() if f.endswith(".json")]
print(f"Found {len(json_files)} JSON files")

# Step 2: Extraction function (FIXED 🔥)
def extract_data(json_data):
    cve = json_data.get("cve", {})
    
    cve_id = cve.get("id", "NA")
    
    description = "NA"
    if cve.get("descriptions"):
        description = cve["descriptions"][0].get("value", "NA")
    
    # CVSS
    cvss = 0
    attack_vector = "NA"
    privileges = "NA"
    
    try:
        cvss_data = cve["metrics"]["cvssMetricV31"][0]["cvssData"]
        cvss = cvss_data.get("baseScore", 0)
        attack_vector = cvss_data.get("attackVector", "NA")
        privileges = cvss_data.get("privilegesRequired", "NA")
    except:
        pass
    
    # Exploit check
    exploit = False
    for ref in cve.get("references", []):
        if "tags" in ref and "Exploit" in ref["tags"]:
            exploit = True
            break
    
    # Label
    if cvss >= 8:
        label = "HIGH"
    elif cvss >= 5:
        label = "MEDIUM"
    else:
        label = "LOW"
    
    rows = []

    # 🔥 FIX: LOOP THROUGH ALL CPE MATCHES
    try:
        configurations = cve.get("configurations", [])
        
        for config in configurations:
            for node in config.get("nodes", []):
                for match in node.get("cpeMatch", []):
                    
                    cpe = match.get("criteria", "")
                    parts = cpe.split(":")
                    
                    if len(parts) > 5:
                        product = parts[4]
                        version = parts[5]
                    else:
                        product = "NA"
                        version = "NA"
                    
                    # Operator logic
                    operator = "="
                    
                    if "versionEndExcluding" in match:
                        version = match["versionEndExcluding"]
                        operator = "<"
                    
                    elif "versionStartIncluding" in match:
                        version = match["versionStartIncluding"]
                        operator = ">="
                    
                    # Create row
                    rows.append({
                        "cve_id": cve_id,
                        "description": description,
                        "cvss": cvss,
                        "attack_vector": attack_vector,
                        "privileges": privileges,
                        "product": product,
                        "version": version,
                        "operator": operator,
                        "exploit": exploit,
                        "label": label
                    })
    
    except Exception as e:
        print(f"Error extracting CPE: {e}")
    
    return rows


# Step 3: Process all files
all_rows = []

for file in json_files:
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)
            rows = extract_data(data)
            all_rows.extend(rows)   # 🔥 important (extend, not append)
    except Exception as e:
        print(f"Error in {file}: {e}")

# Step 4: Convert to DataFrame
df = pd.DataFrame(all_rows)

# Step 5: Save CSV
df.to_csv("dataset_fixed.csv", index=False)

print("✅ Fixed dataset created!")
print(df.head())

Found 27238 JSON files
✅ Fixed dataset created!
          cve_id                                        description  cvss  \
0  CVE-2024-0007  A cross-site scripting (XSS) vulnerability in ...   6.8   
1  CVE-2024-0007  A cross-site scripting (XSS) vulnerability in ...   6.8   
2  CVE-2024-0007  A cross-site scripting (XSS) vulnerability in ...   6.8   
3  CVE-2024-0007  A cross-site scripting (XSS) vulnerability in ...   6.8   
4  CVE-2024-0007  A cross-site scripting (XSS) vulnerability in ...   6.8   

  attack_vector privileges product  version operator  exploit   label  
0       NETWORK       HIGH  pan-os   8.1.24        <    False  MEDIUM  
1       NETWORK       HIGH  pan-os   9.0.17        <    False  MEDIUM  
2       NETWORK       HIGH  pan-os   9.1.16        <    False  MEDIUM  
3       NETWORK       HIGH  pan-os  10.0.11        <    False  MEDIUM  
4       NETWORK       HIGH  pan-os   10.1.6        <    False  MEDIUM  
